In [ ]:
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
from constant import *
from ChatGpt4Model import ChatGpt4Model
from TrainStrategy import TrainStrategy
from LlmOutputLabelConverter import LlmOutputLabelConverter

In [11]:
load_dotenv()
classification_label_set = set(classify_train_dataset['label']) | set(classify_test_dataset['label'])
output_label_converter = LlmOutputLabelConverter(classification_label_set, DEFAULT_CLASSIFICATION_CLASS)

In [12]:
prompt_template = PromptTemplate(
    name="default",
definition = """
You are a Code Analysis Expert specializing in identifying and categorizing Self-Admitted Technical Debt (SATD) in Java test code comments.

A comment is considered SATD if a developer explicitly acknowledges that the code requires future work, improvement, or is relying on a temporary or suboptimal solution.

Your task is to classify each SATD comment into exactly one of the following 15 predefined categories:

Categories of Self-Admitted Technical Debt (SATD):
- build: Issues with the build process, e.g., poorly defined build environment.
- code: Poor naming, ignored exceptions, inefficient/slow algorithms.
- defect: Known unresolved defects, test case failures, unexpected results (without referencing other debt).
- design: Violations of good design principles (e.g., high coupling, low cohesion).
- dependency: Reliance on external dependencies (libraries, APIs, services) that hinder testing.
- documentation: Missing, incomplete, or outdated documentation.
- how-to: Uncertainty or lack of clarity on how to implement/test/resolve an issue; unanswered questions or speculation.
- impractical-case: Explicitly noted problematic cases that should not occur under normal circumstances.
- refactor: Code that requires restructuring (duplication, cleanup, unnecessary code).
- requirement: Incomplete test cases or missing implementation details.
- skip-test: Tests that are skipped, disabled, or intentionally not run.
- subset-test: Only a subset of inputs tested instead of full coverage, often for time/resource reasons.
- superficial-test: Partial or inadequate test coverage.
- temporary-fix: Quick or temporary solutions meant to be replaced later.
- multi: Comment acknowledges more than one type of technical debt.
""",
instruction = "Classify the following test code comment into one of the 15 categories listed above.",
    n_shot_template='Comment: {{ text }}',
    n_shot_answer_template= "Answer: The answer is {{ label }}.",
    line_m_before=3,
    line_n_after=3
)

# Batch Request

In [ ]:
for shots in [0, 15]:
    for model_name in ['gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o-mini', 'gpt-4o']:
        gpt_model = ChatGpt4Model('classify', model_name, output_label_converter, True, True)
        gpt_model.fit(classify_n_shot_dataset)
        gpt_model.submit_batch(classify_test_dataset, DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)

# Simple API Request

In [ ]:
for shots in [0, 15]:
    for model_name in ['gpt-5', 'gpt-5-mini', 'gpt-5-nano', 'gpt-4o-mini', 'gpt-4o']:
        gpt_model = ChatGpt4Model('classify', model_name, output_label_converter, True, True)
        gpt_model.fit(classify_n_shot_dataset)
        gpt_model.predict(classify_test_dataset, DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)

# Dry Run

In [ ]:
for shots in [0]:
    for model_name in ['gpt-5-mini']:
        gpt_model = ChatGpt4Model('classify', model_name, output_label_converter, True, True)
        gpt_model.fit(classify_n_shot_dataset)
        gpt_model.submit_batch(classify_test_dataset.select(range(20)), DATASET_NAME, prompt_template, TrainStrategy.N_SHOT_TOP, shots, verbose=False)